In [ ]:
# ============================================================================
# CONFIGURATION — Edit these values before running
# ============================================================================
PROJECT_ID                            = 'my-project'                                          # GCP project where output tables will be stored
DATASET_ID                            = 'unravel_share_US'                                    # BQ dataset where output tables will be stored
PROJECTS_TABLE                        = 'my-project.unravel_share_US.monitored_projects'      # Fully-qualified BQ table with a project_id column
ERROR_TABLE                           = 'my-project.unravel_share_US.error_log'               # Fully-qualified BQ table to log all errors
PROJECTS_REFRESH_INTERVAL_HOURS       = 24                                                    # How often to refresh project data
SCHEDULED_JOBS_REFRESH_INTERVAL_HOURS = 24                                                    # How often to refresh scheduled job data
# ============================================================================


import re
import json
import time
import traceback
import concurrent.futures
from datetime import datetime, timezone

import requests
import google.auth
import google.auth.transport.requests
from google.cloud import bigquery
from google.cloud import resourcemanager_v3
from google.api_core.exceptions import NotFound


# Validate dataset_id
if not re.match(r'^[A-Za-z_][A-Za-z0-9_]{0,1023}$', DATASET_ID):
    raise ValueError(
        f"Invalid DATASET_ID '{DATASET_ID}'. "
        "Must start with a letter or underscore, letters/digits/underscores only, no hyphens."
    )

print("Configuration:")
print(f"  PROJECT_ID                            = {PROJECT_ID}")
print(f"  DATASET_ID                            = {DATASET_ID}")
print(f"  PROJECTS_TABLE                        = {PROJECTS_TABLE}")
print(f"  ERROR_TABLE                           = {ERROR_TABLE}")
print(f"  PROJECTS_REFRESH_INTERVAL_HOURS       = {PROJECTS_REFRESH_INTERVAL_HOURS}")
print(f"  SCHEDULED_JOBS_REFRESH_INTERVAL_HOURS = {SCHEDULED_JOBS_REFRESH_INTERVAL_HOURS}")


# ============================================================================
# Auth
# ============================================================================
def get_credentials():
    credentials, _ = google.auth.default()
    credentials.refresh(google.auth.transport.requests.Request())
    return credentials


# ============================================================================
# Error Logger
# ============================================================================
# Shared run timestamp — same value across all error rows in a single execution
RUN_TS = datetime.now(timezone.utc).isoformat()

def log_error(client, dest_table, project_id, error_message, failed_sql=None):
    """
    Insert one row into ERROR_TABLE. Never raises — if the insert itself
    fails, the error is printed to stdout so the run can continue.
    """
    now = datetime.now(timezone.utc).isoformat()
    row = {
        'run_ts':        RUN_TS,
        'dest_table':    dest_table,
        'project_id':    project_id,
        'error_message': str(error_message),
        'failed_sql':    failed_sql,
        'logged_at':     now,
    }
    print(f"  [ERROR] {dest_table} | {project_id} | {error_message}")
    try:
        errs = client.insert_rows_json(ERROR_TABLE, [row])
        if errs:
            print(f"  [WARN] Could not write to error_log: {errs}")
    except Exception as exc:
        print(f"  [WARN] Exception writing to error_log: {exc}")


# ============================================================================
# API Fetchers
# ============================================================================
def fetch_paginated(url, credentials, result_key):
    all_items, params = [], {}
    while True:
        resp = requests.get(url, params=params,
                            headers={'Authorization': f'Bearer {credentials.token}'})
        if resp.status_code != 200:
            return None, f"GET {url} returned {resp.status_code}: {resp.text}"
        body = resp.json()
        all_items.extend(body.get(result_key, []))
        next_token = body.get('nextPageToken')
        if not next_token:
            break
        params['pageToken'] = next_token
        print(f"  Fetching next page (token={next_token[:12]}...)")
    return all_items, None


def fetch_project_api_data(client, project):
    """
    Calls GCP APIs for a single project.
    API-level errors are logged to ERROR_TABLE and the affected key is omitted
    from the result — other projects continue unaffected.
    """
    creds = get_credentials()
    print(f"Fetching API data for project: {project}")
    base = 'https://cloudresourcemanager.googleapis.com/v1/projects'
    result = {}

    # Project details
    try:
        proj_resp = requests.get(f"{base}/{project}",
                                 headers={'Authorization': f'Bearer {creds.token}'})
        if proj_resp.status_code == 200:
            result['projectDetails'] = proj_resp.json()
        else:
            log_error(client, 'project_details', project,
                      f"projectDetails API returned {proj_resp.status_code}: {proj_resp.text}",
                      f"GET {base}/{project}")
    except Exception as exc:
        log_error(client, 'project_details', project,
                  f"projectDetails API exception: {exc}\n{traceback.format_exc()}")

    # Scheduled jobs
    try:
        jobs_url = f'https://bigquerydatatransfer.googleapis.com/v1/projects/{project}/transferConfigs'
        jobs, err = fetch_paginated(jobs_url, creds, 'transferConfigs')
        if err:
            log_error(client, 'scheduled_job_details', project,
                      f"transferConfigs API error: {err}", jobs_url)
        else:
            result['scheduledJobDetails'] = jobs
    except Exception as exc:
        log_error(client, 'scheduled_job_details', project,
                  f"transferConfigs API exception: {exc}\n{traceback.format_exc()}")

    print(f"  Fetch complete for: {project}")
    return result


# ============================================================================
# Transformers
# ============================================================================
def transform_project(raw):
    return {
        'project_number':  raw.get('projectNumber'),
        'project_id':      raw.get('projectId'),
        'lifecycle_state': raw.get('lifecycleState'),
        'name':            raw.get('name'),
        'create_time':     raw.get('createTime'),
        'labels':          json.dumps(raw['labels']) if raw.get('labels') else None,
        'parent':          raw.get('parent'),
        'tags':            json.dumps(raw['tags'])   if raw.get('tags')   else None,
    }


def transform_jobs(project, jobs):
    rows = []
    for cfg in jobs:
        so    = cfg.get('scheduleOptions')
        so_v2 = cfg.get('scheduleOptionsV2')
        ep    = cfg.get('emailPreferences')
        enc   = cfg.get('encryptionConfiguration')
        rows.append({
            'project':        project,
            'name':           cfg.get('name'),
            'display_name':   cfg.get('displayName'),
            'data_source_id': cfg.get('dataSourceId'),
            'params':         json.dumps(cfg['params']) if cfg.get('params') else None,
            'schedule':       cfg.get('schedule'),
            'schedule_options': None if not so else {
                'disable_auto_scheduling': so.get('disableAutoScheduling'),
                'start_time': so.get('startTime'),
                'end_time':   so.get('endTime'),
            },
            'schedule_options_V2': None if not so_v2 else {
                'time_based_schedule': None if not so_v2.get('timeBasedSchedule') else {
                    'schedule':   so_v2['timeBasedSchedule'].get('schedule'),
                    'start_time': so_v2['timeBasedSchedule'].get('startTime'),
                    'end_time':   so_v2['timeBasedSchedule'].get('endTime'),
                },
                'manual_schedule':       json.dumps(so_v2['manualSchedule']) if so_v2.get('manualSchedule') else None,
                'event_driven_schedule': None if not so_v2.get('eventDrivenSchedule') else {
                    'pubsub_subscription': so_v2['eventDrivenSchedule'].get('pubsubSubscription'),
                },
            },
            'data_refresh_window_days':  cfg.get('dataRefreshWindowDays'),
            'disabled':                  cfg.get('disabled'),
            'update_time':               cfg.get('updateTime'),
            'next_run_time':             cfg.get('nextRunTime'),
            'state':                     cfg.get('state'),
            'user_id':                   cfg.get('userId'),
            'dataset_region':            cfg.get('datasetRegion'),
            'notification_pubsub_topic': cfg.get('notificationPubsubTopic'),
            'email_preferences':         None if not ep  else {'enable_failure_email': ep.get('enableFailureEmail')},
            'encryption_configuration':  None if not enc else {'kms_key_name': enc.get('kmsKeyName')},
            'error':                     json.dumps(cfg['error']) if cfg.get('error') else None,
            'destination_dataset_id':    cfg.get('destinationDatasetId'),
            'owner_info':                cfg.get('ownerInfo'),
        })
    return rows


# ============================================================================
# BQ Helpers
# ============================================================================
def bq_run(client, sql, dest_table=None, project_id=None):
    """
    Execute a SQL statement. On failure, log to ERROR_TABLE and return False.
    Returns the query result on success, False on failure.
    """
    try:
        return client.query(sql).result()
    except Exception as exc:
        log_error(client, dest_table or 'unknown', project_id or 'N/A',
                  f"{exc}\n{traceback.format_exc()}", sql)
        return False


def table_exists(client, full_id):
    try:
        client.get_table(full_id)
        return True
    except NotFound:
        return False


def truncate(client, full_id):
    print(f"  Truncating {full_id} ...")
    return bq_run(client, f"TRUNCATE TABLE `{full_id}`", dest_table=full_id)


def drop_table(client, full_id):
    """Drop a table if it exists — used for temp table cleanup after each run."""
    try:
        client.delete_table(full_id)
        print(f"  Dropped temp table: {full_id}")
    except NotFound:
        pass  # already gone, nothing to do
    except Exception as exc:
        log_error(client, full_id, 'N/A', f"Could not drop temp table: {exc}\n{traceback.format_exc()}")


def should_refresh(client, full_id, interval_hours):
    try:
        tbl = client.get_table(full_id)
        if tbl.num_rows == 0:
            print(f"  {full_id} is empty -> refresh needed.")
            return True
        if tbl.modified is None:
            return True
        elapsed = (datetime.now(timezone.utc) - tbl.modified).total_seconds() / 3600
        if elapsed >= interval_hours:
            print(f"  {full_id} last modified {elapsed:.1f}h ago (threshold {interval_hours}h) -> refresh needed.")
            return True
        print(f"  {full_id} last modified {elapsed:.1f}h ago (threshold {interval_hours}h) -> skipping.")
        return False
    except NotFound:
        return True
    except Exception as exc:
        log_error(client, full_id, 'N/A',
                  f"could not check refresh status, defaulting to refresh: {exc}\n{traceback.format_exc()}")
        return True


def copy_staging_to_target(client, staging, target):
    result = bq_run(client, f"SELECT COUNT(*) AS cnt FROM `{staging}`", dest_table=staging)
    if result is False:
        return False
    row_count = list(result)[0].cnt
    if row_count == 0:
        print(f"  Staging {staging} is empty - skipping copy to preserve {target}.")
        return False
    print(f"  Copying {row_count} rows: {staging} -> {target}")
    if truncate(client, target) is False:
        return False
    sql = f"INSERT INTO `{target}` SELECT * FROM `{staging}`"
    if bq_run(client, sql, dest_table=target) is False:
        return False
    print("  Copy complete.")
    return True


# ============================================================================
# DDL
# ============================================================================
SCHEDULED_JOB_COLS = """
    project STRING, name STRING, display_name STRING, data_source_id STRING,
    params JSON, schedule STRING,
    schedule_options STRUCT<disable_auto_scheduling BOOLEAN, start_time STRING, end_time STRING>,
    schedule_options_V2 STRUCT<
        time_based_schedule STRUCT<schedule STRING, start_time STRING, end_time STRING>,
        manual_schedule JSON,
        event_driven_schedule STRUCT<pubsub_subscription STRING>
    >,
    data_refresh_window_days INTEGER, disabled BOOLEAN,
    update_time STRING, next_run_time STRING, state STRING, user_id STRING,
    dataset_region STRING, notification_pubsub_topic STRING,
    email_preferences STRUCT<enable_failure_email BOOLEAN>,
    encryption_configuration STRUCT<kms_key_name STRING>,
    error STRING, destination_dataset_id STRING,
    owner_info STRUCT<email STRING>
"""

# error_log uses its own partitioned/clustered DDL; project_details and
# scheduled_job_details are plain tables. Temp table is created per-run.
CREATE_TABLE_STATEMENTS = [
    f"""CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.project_details` (
        project_number STRING, project_id STRING, lifecycle_state STRING,
        name STRING, create_time STRING, labels JSON,
        parent STRUCT<type STRING, id STRING>, tags JSON)""",

    f"CREATE TABLE IF NOT EXISTS `{PROJECT_ID}.{DATASET_ID}.scheduled_job_details` ({SCHEDULED_JOB_COLS})",

    f"""CREATE TABLE IF NOT EXISTS `{ERROR_TABLE}` (
        run_ts        TIMESTAMP,
        dest_table    STRING,
        project_id    STRING,
        error_message STRING,
        failed_sql    STRING,
        logged_at     TIMESTAMP
    )
    PARTITION BY DATE(logged_at)
    CLUSTER BY project_id""",
]


def create_tables(client):
    print("\n## Ensuring output tables exist ##")
    with concurrent.futures.ThreadPoolExecutor() as pool:
        futures = {pool.submit(bq_run, client, sql): sql for sql in CREATE_TABLE_STATEMENTS}
        for fut in concurrent.futures.as_completed(futures):
            result = fut.result()
            match  = re.search(r'`([^`]+)`', futures[fut])
            name   = match.group(1) if match else '?'
            if result is False:
                print(f"  [ERROR] Could not create table: {name}")
            else:
                print(f"  OK: {name}")


def verify_tables(client):
    # error_log is in ERROR_TABLE (may be a different project/dataset)
    tables_to_verify = [
        f"{PROJECT_ID}.{DATASET_ID}.project_details",
        f"{PROJECT_ID}.{DATASET_ID}.scheduled_job_details",
        ERROR_TABLE,
    ]
    time.sleep(5)
    for full in tables_to_verify:
        for attempt in range(6):
            if table_exists(client, full):
                print(f"  Verified: {full}")
                break
            print(f"  Waiting for {full} (attempt {attempt+1}/6)...")
            time.sleep(5)
        else:
            print(f"  [ERROR] Table {full} not available after retries.")
            return False
    return True


# ============================================================================
# Bootstrap Checks
# ============================================================================
def project_exists(pid):
    rm = resourcemanager_v3.ProjectsClient()
    try:
        rm.get_project(name=f'projects/{pid}')
        return True
    except NotFound:
        print(f"Project '{pid}' not found.")
        return False
    except Exception as exc:
        print(f"Error checking project '{pid}': {exc}")
        return False


def ensure_dataset(client):
    full = f"{PROJECT_ID}.{DATASET_ID}"
    try:
        client.get_dataset(full)
        print(f"Dataset {full} already exists.")
        return True
    except NotFound:
        print(f"Dataset {full} not found - creating ...")
        ds = bigquery.Dataset(full)
        ds.location = 'US'
        client.create_dataset(ds)
        print(f"Dataset {full} created.")
        return True
    except Exception as exc:
        print(f"[ERROR] checking/creating dataset: {exc}")
        return False


# ============================================================================
# Main ETL
# ============================================================================
def store_to_bq(client, projects):
    tbl     = lambda name: f"{PROJECT_ID}.{DATASET_ID}.{name}"
    tbl_tmp = lambda name: f"{PROJECT_ID}.{DATASET_ID}.{name}_temp"

    refresh_projects  = should_refresh(client, tbl('project_details'),       PROJECTS_REFRESH_INTERVAL_HOURS)
    refresh_scheduled = refresh_projects or should_refresh(
        client, tbl('scheduled_job_details'), SCHEDULED_JOBS_REFRESH_INTERVAL_HOURS
    )

    if not any([refresh_projects, refresh_scheduled]):
        print("All tables are within their refresh interval. Nothing to do.")
        return

    # Create temp table fresh at the start of each run.
    # Drop first in case a previous run crashed before cleanup.
    if refresh_scheduled:
        drop_table(client, tbl_tmp('scheduled_job_details'))
        bq_run(client,
               f"CREATE TABLE `{tbl_tmp('scheduled_job_details')}` ({SCHEDULED_JOB_COLS})",
               dest_table=tbl_tmp('scheduled_job_details'))
        print(f"  Created temp table: {tbl_tmp('scheduled_job_details')}")

    all_project_rows = []

    try:
        for project in projects:
            data = fetch_project_api_data(client, project)

            # --- project_details ---
            if refresh_projects and 'projectDetails' in data:
                try:
                    row = transform_project(data['projectDetails'])
                    if row:
                        all_project_rows.append(row)
                except Exception as exc:
                    log_error(client, 'project_details', project,
                              f"transform_project failed: {exc}\n{traceback.format_exc()}")

            # --- scheduled_job_details ---
            if refresh_scheduled and data.get('scheduledJobDetails'):
                try:
                    rows = transform_jobs(project, data['scheduledJobDetails'])
                    if rows:
                        errs = client.insert_rows_json(tbl_tmp('scheduled_job_details'), rows)
                        if errs:
                            log_error(client, tbl_tmp('scheduled_job_details'), project,
                                      f"insert_rows_json errors: {errs}")
                        else:
                            print(f"  {len(rows)} scheduled job row(s) staged for {project}.")
                except Exception as exc:
                    log_error(client, 'scheduled_job_details', project,
                              f"transform/insert failed: {exc}\n{traceback.format_exc()}")

        # --- Flush project_details (in-memory, no temp table) ---
        if refresh_projects:
            if all_project_rows:
                if truncate(client, tbl('project_details')) is not False:
                    errs = client.insert_rows_json(tbl('project_details'), all_project_rows)
                    if errs:
                        log_error(client, 'project_details', 'N/A',
                                  f"insert_rows_json errors: {errs}")
                    else:
                        print(f"  {len(all_project_rows)} project rows inserted.")
            else:
                print("  No project data fetched - skipping project_details truncate.")

        # --- Copy staging -> target for scheduled jobs ---
        if refresh_scheduled:
            print("\nWaiting 30s for streaming buffer to settle ...")
            time.sleep(30)
            copy_staging_to_target(client,
                                   tbl_tmp('scheduled_job_details'),
                                   tbl('scheduled_job_details'))

    finally:
        # Always drop temp tables — even if the run failed midway
        print("\n## Cleaning up temp tables ##")
        if refresh_scheduled:
            drop_table(client, tbl_tmp('scheduled_job_details'))


# ============================================================================
# Run
# ============================================================================
client = bigquery.Client(project=PROJECT_ID)

print(f"\nFetching project list from `{PROJECTS_TABLE}` ...")
try:
    rows = list(
        client.query(
            f"SELECT DISTINCT project_id FROM `{PROJECTS_TABLE}` WHERE project_id IS NOT NULL"
        ).result()
    )
    projects_to_fetch = [r.project_id for r in rows]
    if not projects_to_fetch:
        raise ValueError("No project IDs found in the supplied PROJECTS_TABLE.")
    print(f"Found {len(projects_to_fetch)} project(s): {projects_to_fetch}")
except Exception as exc:
    raise SystemExit(f"[ERROR] fetching project list: {exc}") from exc

if not project_exists(PROJECT_ID):
    raise SystemExit(1)
if not ensure_dataset(client):
    raise SystemExit(1)

create_tables(client)

if not verify_tables(client):
    raise SystemExit(1)

store_to_bq(client, projects_to_fetch)
print("\nDone.")